# Graphical Models Practice Notebook

This notebook is a hands-on companion to the Markdown file on **Graphical Models**.
It demonstrates probability factorization, inference, and approximate sampling.

Topics covered:

1. DAG factorization
2. Conditional independence intuition
3. d-Separation examples
4. Bayesian network inference
5. Markov Random Field potentials
6. Gibbs sampling
7. Hidden Markov Model simulation
8. Viterbi-style decoding intuition
9. Summary table
10. Mini exercises

The notebook is designed for learning, GitHub repositories, and classroom use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(42)

## 1. DAG Factorization: A -> B -> C

In a Directed Acyclic Graph (DAG), the joint distribution factorizes as:

$$P(X_1, ..., X_n) = \prod_{i=1}^{n} P(X_i \mid Pa(X_i))$$

For the chain A -> B -> C:

$$P(A, B, C) = P(A) \cdot P(B|A) \cdot P(C|B)$$

In [ ]:
P_A = {0: 0.4, 1: 0.6}
P_B_given_A = {(0, 0): 0.7, (1, 0): 0.3, (0, 1): 0.2, (1, 1): 0.8}   # (B, A)
P_C_given_B = {(0, 0): 0.9, (1, 0): 0.1, (0, 1): 0.25, (1, 1): 0.75} # (C, B)

rows = []
for A in [0, 1]:
    for B in [0, 1]:
        for C in [0, 1]:
            p = P_A[A] * P_B_given_A[(B, A)] * P_C_given_B[(C, B)]
            rows.append((A, B, C, p))

joint = pd.DataFrame(rows, columns=['A', 'B', 'C', 'P(A,B,C)'])
print(f'Sum of joint probabilities: {joint["P(A,B,C)"].sum():.4f}')
joint

## 2. Conditional Independence Intuition

In a chain A -> B -> C, A and C are **marginally dependent** but **conditionally independent given B**:

$$A \perp C \mid B$$

We verify this by computing P(C=1|A) with and without conditioning on B.

In [ ]:
# Marginal P(C=1 | A)
p_ac = joint.groupby(['A', 'C'])['P(A,B,C)'].sum().reset_index()
p_a  = joint.groupby('A')['P(A,B,C)'].sum().reset_index().rename(columns={'P(A,B,C)': 'P(A)'})
p_ac = p_ac.merge(p_a, on='A')
p_ac['P(C|A)'] = p_ac['P(A,B,C)'] / p_ac['P(A)']
print('P(C=1|A) — A and C are marginally dependent:')
print(p_ac[p_ac['C'] == 1][['A', 'P(C|A)']].to_string(index=False))

# Conditional P(C=1 | A, B) vs P(C=1 | B) — should be the same
p_bc = joint.groupby(['B', 'C'])['P(A,B,C)'].sum().reset_index()
p_b  = joint.groupby('B')['P(A,B,C)'].sum().reset_index().rename(columns={'P(A,B,C)': 'P(B)'})
p_bc = p_bc.merge(p_b, on='B')
p_bc['P(C|B)'] = p_bc['P(A,B,C)'] / p_bc['P(B)']
print('\nP(C=1|B) — once we know B, A adds no information about C:')
print(p_bc[p_bc['C'] == 1][['B', 'P(C|B)']].to_string(index=False))

## 3. d-Separation: Three Patterns

d-Separation determines conditional independence from graph structure.

| Pattern | Graph | Conditioning on Z | Result |
|---------|-------|-------------------|--------|
| **Chain** | X -> Z -> Y | Yes | Blocks path: X \perp Y \| Z |
| **Fork** | X <- Z -> Y | Yes | Blocks path: X \perp Y \| Z |
| **Collider** | X -> Z <- Y | Yes | **Opens** path: X and Y become dependent |

The collider case is counterintuitive: conditioning on Z creates dependence between X and Y.

In [ ]:
n = 5000

# --- Chain: X -> Z -> Y ---
X_chain = np.random.normal(0, 1, n)
Z_chain = 0.8 * X_chain + np.random.normal(0, 0.5, n)
Y_chain = 0.8 * Z_chain + np.random.normal(0, 0.5, n)

corr_XY_marginal = np.corrcoef(X_chain, Y_chain)[0, 1]
# Partial correlation controlling Z
resid_X = X_chain - LinearRegression_simple(Z_chain, X_chain)
resid_Y = Y_chain - LinearRegression_simple(Z_chain, Y_chain)

def partial_corr_via_residuals(X, Y, Z):
    """Partial correlation of X and Y given Z."""
    rXZ = np.corrcoef(X, Z)[0, 1]
    rYZ = np.corrcoef(Y, Z)[0, 1]
    rXY = np.corrcoef(X, Y)[0, 1]
    return (rXY - rXZ*rYZ) / (np.sqrt(1 - rXZ**2) * np.sqrt(1 - rYZ**2))

# Chain
corr_XY_given_Z_chain = partial_corr_via_residuals(X_chain, Y_chain, Z_chain)

# Fork: X <- Z -> Y
Z_fork = np.random.normal(0, 1, n)
X_fork = 0.8 * Z_fork + np.random.normal(0, 0.5, n)
Y_fork = 0.8 * Z_fork + np.random.normal(0, 0.5, n)
corr_XY_fork_marginal = np.corrcoef(X_fork, Y_fork)[0, 1]
corr_XY_fork_given_Z  = partial_corr_via_residuals(X_fork, Y_fork, Z_fork)

# Collider: X -> Z <- Y
X_col = np.random.normal(0, 1, n)
Y_col = np.random.normal(0, 1, n)
Z_col = 0.6 * X_col + 0.6 * Y_col + np.random.normal(0, 0.3, n)
corr_XY_col_marginal = np.corrcoef(X_col, Y_col)[0, 1]
corr_XY_col_given_Z  = partial_corr_via_residuals(X_col, Y_col, Z_col)

pd.DataFrame({
    'Pattern':              ['Chain X->Z->Y', 'Fork X<-Z->Y', 'Collider X->Z<-Y'],
    'Corr(X,Y) marginal':   [corr_XY_marginal, corr_XY_fork_marginal, corr_XY_col_marginal],
    'Corr(X,Y) | Z':        [corr_XY_given_Z_chain, corr_XY_fork_given_Z, corr_XY_col_given_Z]
})

**Observation:** For the collider, marginal Corr(X,Y) \approx 0 (independent), but
conditioning on Z creates a spurious correlation — this is collider bias.

## 4. Bayesian Network Inference: P(C=1)

We compute the marginal probability of C=1 by summing over all joint configurations.

In [ ]:
p_c1 = joint.loc[joint['C'] == 1, 'P(A,B,C)'].sum()

# P(A=1 | C=1) via Bayes
p_a1_c1 = joint.loc[(joint['A'] == 1) & (joint['C'] == 1), 'P(A,B,C)'].sum() / p_c1

pd.DataFrame({
    'Query':  ['P(C=1)', 'P(A=1 | C=1)'],
    'Value':  [p_c1, p_a1_c1]
})

## 5. Markov Random Field Potentials

Undirected graphical models define the joint through potential functions:

$$P(X) = \frac{1}{Z} \prod_c \phi_c(X_c)$$

Nodes with the same state are encouraged (higher potential).

In [ ]:
def phi(x, y):
    return 3 if x == y else 1

states = [(0, 0), (0, 1), (1, 0), (1, 1)]
mrf_rows, Z_norm = [], 0
for x1, x2 in states:
    pot = phi(x1, x2)
    Z_norm += pot
    mrf_rows.append((x1, x2, pot))

mrf = pd.DataFrame(mrf_rows, columns=['X1', 'X2', 'Potential'])
mrf['Probability'] = mrf['Potential'] / Z_norm
mrf

## 6. Gibbs Sampling

Gibbs sampling is an MCMC method for MRFs and other intractable distributions.
It samples each variable from its conditional distribution given all others.

In [ ]:
gibbs_samples = []
x1, x2 = 0, 0

for _ in range(3000):
    p_x1_1 = phi(1, x2) / (phi(0, x2) + phi(1, x2))
    x1 = np.random.binomial(1, p_x1_1)
    p_x2_1 = phi(x1, 1) / (phi(x1, 0) + phi(x1, 1))
    x2 = np.random.binomial(1, p_x2_1)
    gibbs_samples.append((x1, x2))

gibbs_df = pd.DataFrame(gibbs_samples, columns=['X1', 'X2'])
empirical = gibbs_df.value_counts(normalize=True).reset_index(name='Empirical prob')
result = empirical.merge(mrf[['X1', 'X2', 'Probability']], on=['X1', 'X2'])
result.rename(columns={'Probability': 'True prob'}, inplace=True)
result

## 7. Hidden Markov Model Simulation

An HMM has hidden states Z_t and observations X_t:
- **Transition:** P(Z_t | Z_{t-1})
- **Emission:** P(X_t | Z_t)

The joint:

$$P(Z_{1:T}, X_{1:T}) = P(Z_1)\prod_{t=2}^{T}P(Z_t|Z_{t-1})\prod_{t=1}^{T}P(X_t|Z_t)$$

In [ ]:
T_hmm = 60
# Transition probabilities: P(Z_t=1 | Z_{t-1})
trans = {0: 0.3, 1: 0.8}   # tends to stay in current state
# Emission probabilities: P(X_t=1 | Z_t)
emit  = {0: 0.15, 1: 0.85}

z_states, x_obs = [], []
z = 0
for t in range(T_hmm):
    z = np.random.binomial(1, trans[z])
    x = np.random.binomial(1, emit[z])
    z_states.append(z)
    x_obs.append(x)

hmm_df = pd.DataFrame({'t': np.arange(T_hmm), 'hidden_state': z_states, 'observation': x_obs})

plt.figure(figsize=(11, 4))
plt.plot(hmm_df['t'], hmm_df['hidden_state'], label='Hidden state', linewidth=2)
plt.plot(hmm_df['t'], hmm_df['observation'], label='Observation', linestyle='--', alpha=0.7)
plt.title('HMM: Hidden State vs Observation')
plt.xlabel('Time')
plt.ylabel('State / Observation')
plt.legend()
plt.show()

pd.DataFrame({
    'Metric':  ['Fraction of time in state 1 (true)', 'Fraction of X=1 observations'],
    'Value':   [np.mean(z_states), np.mean(x_obs)]
})

## 8. Viterbi-Style Decoding Intuition

The Viterbi algorithm finds the most likely hidden state sequence given observations.

It uses dynamic programming:

$$\delta_t(j) = \max_i \delta_{t-1}(i) \cdot P(Z_t=j|Z_{t-1}=i) \cdot P(X_t|Z_t=j)$$

Below we implement a simplified two-state Viterbi decoder.

In [ ]:
def viterbi_two_state(observations, trans_probs, emit_probs, pi):
    """trans_probs[i,j] = P(Z_t=j|Z_{t-1}=i), emit_probs[s][x] = P(X=x|Z=s)"""
    T = len(observations)
    n_states = 2
    delta = np.zeros((T, n_states))
    psi   = np.zeros((T, n_states), dtype=int)

    delta[0] = [pi[s] * emit_probs[s][observations[0]] for s in range(n_states)]

    for t in range(1, T):
        for j in range(n_states):
            scores = [delta[t-1, i] * trans_probs[i, j] * emit_probs[j][observations[t]]
                      for i in range(n_states)]
            delta[t, j] = max(scores)
            psi[t, j]   = int(np.argmax(scores))

    # Backtrack
    path = [int(np.argmax(delta[-1]))]
    for t in range(T-1, 0, -1):
        path.insert(0, psi[t, path[0]])
    return np.array(path)

trans_matrix = np.array([[1 - trans[0], trans[0]],
                          [1 - trans[1], trans[1]]])
emit_matrix  = [{0: 1 - emit[s], 1: emit[s]} for s in [0, 1]]
pi_init      = [0.5, 0.5]

viterbi_path = viterbi_two_state(x_obs, trans_matrix, emit_matrix, pi_init)
accuracy = np.mean(np.array(viterbi_path) == np.array(z_states))

pd.DataFrame({
    'Metric':  ['Viterbi state accuracy', 'T', 'Most frequent decoded state'],
    'Value':   [round(accuracy, 4), T_hmm, int(np.bincount(viterbi_path).argmax())]
})

## 9. Summary Table

In [ ]:
summary = pd.DataFrame({
    'Measure': [
        'P(C=1) marginal',
        'P(A=1 | C=1)',
        'Most likely MRF state (X1,X2)',
        'Gibbs empirical P(1,1)',
        'HMM mean hidden state',
        'HMM mean observation',
        'Viterbi decoding accuracy'
    ],
    'Value': [
        round(p_c1, 4),
        round(p_a1_c1, 4),
        '(1,1)',
        round(gibbs_df[(gibbs_df.X1 == 1) & (gibbs_df.X2 == 1)].shape[0] / len(gibbs_df), 4),
        round(np.mean(z_states), 4),
        round(np.mean(x_obs), 4),
        round(accuracy, 4)
    ]
})
summary

## 10. Mini Exercises

Try these on your own:

1. Change the DAG structure from A -> B -> C to a fork A <- B -> C and recompute the joint distribution.
2. In the d-separation cell, increase the collider coefficient and measure how strong the spurious correlation becomes.
3. Extend the BN to a four-node DAG (e.g., add node D depending on C) and compute P(D=1).
4. Change the MRF potential to phi(x,y) = 5 when x == y and recompute the probabilities.
5. Run Gibbs sampling for 10,000 iterations and compare empirical vs exact probabilities.
6. Modify HMM transition probabilities to make the chain switch states more frequently and observe how observation alignment changes.
7. Compare Viterbi decoding accuracy for different emission probabilities (well-separated vs overlapping).
8. Implement a three-state HMM and visualize the state sequence and observations.

These exercises are especially useful for probabilistic reasoning, AI systems, NLP, bioinformatics, and sequential decision making.